In [1]:
import os
import pandas as pd
import tensorflow as tf
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam

2026-05-03 12:34:50.402062: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
tf.config.list_physical_devices("GPU")

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [3]:
PROJECT_ROOT = "/home/jovyan"

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

metadata_path = os.path.join(DATA_DIR, "HAM10000_metadata.csv")
metadata = pd.read_csv(metadata_path)

print(metadata.shape)
print(metadata["dx"].value_counts())

(10015, 7)
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64


In [4]:
IMG_DIR_1 = os.path.join(DATA_DIR, "HAM10000_images_part_1")
IMG_DIR_2 = os.path.join(DATA_DIR, "HAM10000_images_part_2")

def get_image_path(image_id):
    path1 = os.path.join(IMG_DIR_1, image_id + ".jpg")
    if os.path.exists(path1):
        return path1
    return os.path.join(IMG_DIR_2, image_id + ".jpg")

metadata["image_path"] = metadata["image_id"].apply(get_image_path)

metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,image_path
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,/home/jovyan/data/HAM10000_images_part_1/ISIC_...
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,/home/jovyan/data/HAM10000_images_part_1/ISIC_...
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,/home/jovyan/data/HAM10000_images_part_1/ISIC_...
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,/home/jovyan/data/HAM10000_images_part_1/ISIC_...
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,/home/jovyan/data/HAM10000_images_part_2/ISIC_...


In [5]:
print(metadata["image_path"].iloc[0])
print(os.path.exists(metadata["image_path"].iloc[0]))

/home/jovyan/data/HAM10000_images_part_1/ISIC_0027419.jpg
True


In [6]:
metadata["dx"].value_counts()

dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

In [7]:
IMG_DIR_1 = os.path.join(DATA_DIR, "HAM10000_images_part_1")
IMG_DIR_2 = os.path.join(DATA_DIR, "HAM10000_images_part_2")

print(len([f for f in os.listdir(IMG_DIR_1) if f.endswith(".jpg")]))
print(len([f for f in os.listdir(IMG_DIR_2) if f.endswith(".jpg")]))

5000
5015


In [9]:
# Paths for Jupyter / GPUHub
PROJECT_ROOT = "/home/jovyan"

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 7
EPOCHS = 20

In [10]:
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("MODELS_DIR:", MODELS_DIR)
print(os.listdir(DATA_DIR))

PROJECT_ROOT: /home/jovyan
DATA_DIR: /home/jovyan/data
RESULTS_DIR: /home/jovyan/results
MODELS_DIR: /home/jovyan/models
['.ipynb_checkpoints', 'hmnist_28_28_L.csv', 'hmnist_28_28_RGB.csv', 'HAM10000_images_part_1', 'hmnist_8_8_RGB.csv', 'hmnist_8_8_L.csv', 'HAM10000_images_part_2', 'HAM10000_metadata.csv']


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

metadata_path = os.path.join(DATA_DIR, "HAM10000_metadata.csv")
metadata = pd.read_csv(metadata_path)

# image paths
IMG_DIR_1 = os.path.join(DATA_DIR, "HAM10000_images_part_1")
IMG_DIR_2 = os.path.join(DATA_DIR, "HAM10000_images_part_2")

def get_image_path(image_id):
    path1 = os.path.join(IMG_DIR_1, image_id + ".jpg")
    if os.path.exists(path1):
        return path1
    return os.path.join(IMG_DIR_2, image_id + ".jpg")

metadata["image_path"] = metadata["image_id"].apply(get_image_path)

# labels
le = LabelEncoder()
metadata["label"] = le.fit_transform(metadata["dx"])

# split: 70% train, 15% val, 15% test
train_df, temp_df = train_test_split(
    metadata,
    test_size=0.30,
    stratify=metadata["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

# save
train_df.to_csv(os.path.join(RESULTS_DIR, "train_split.csv"), index=False)
val_df.to_csv(os.path.join(RESULTS_DIR, "val_split.csv"), index=False)
test_df.to_csv(os.path.join(RESULTS_DIR, "test_split.csv"), index=False)

print(train_df.shape, val_df.shape, test_df.shape)
print(os.listdir(RESULTS_DIR))

(7010, 9) (1502, 9) (1503, 9)
['old splits', '.ipynb_checkpoints', 'train_split.csv', 'val_split.csv', 'test_split.csv']


In [12]:
# Load split CSVs
train_df = pd.read_csv(os.path.join(RESULTS_DIR, "train_split.csv"))
val_df = pd.read_csv(os.path.join(RESULTS_DIR, "val_split.csv"))

In [13]:
# Compute class weights
classes = np.sort(train_df["label"].unique())

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"]
)

# Convert array to dictionary + cap weights at 4.0
class_weights = {
    int(cls): min(float(weight), 4.0)
    for cls, weight in zip(classes, class_weights_array)
}

print("Class weights:")
for k, v in class_weights.items():
    print(f"Class {k}: {v:.4f}")

Class weights:
Class 0: 4.0000
Class 1: 2.7817
Class 2: 1.3022
Class 3: 4.0000
Class 4: 1.2855
Class 5: 0.2134
Class 6: 4.0000


In [14]:
# Image loader
def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = image / 255.0
    return image, label

In [15]:
# Build datasets
train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["image_path"].values, train_df["label"].values)
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (val_df["image_path"].values, val_df["label"].values)
)

train_ds = (
    train_ds
    .map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_ds
    .map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

I0000 00:00:1777811754.417531     332 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13294 MB memory:  -> device: 0, name: NVIDIA A16, pci bus id: 0000:e5:00.0, compute capability: 8.6


In [16]:
# Data augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.03),
])

In [17]:
# Base model
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

In [18]:
# Build ResNet50 model cleanly

inputs = tf.keras.Input(shape=(224, 224, 3))

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_tensor=inputs
)

# Freeze all layers first
for layer in base_model.layers:
    layer.trainable = False

# Unfreeze last 10 layers
for layer in base_model.layers[-10:]:
    layer.trainable = True

# Head
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 24,147,079 (92.11 MB)

 Trainable params: 5,024,519 (19.17 MB)

 Non-trainable params: 19,122,560 (72.95 MB)

In [19]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(MODELS_DIR, "best_model.keras"),
        save_best_only=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks
)

model.save(os.path.join(MODELS_DIR, "resnet50_finetuned.keras"))

Epoch 1/15


2026-05-03 12:36:21.328378: I external/local_xla/xla/service/service.cc:163] XLA service 0x7f7e240023d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-05-03 12:36:21.328425: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA A16, Compute Capability 8.6
2026-05-03 12:36:24.638186: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-05-03 12:36:29.367347: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900
2026-05-03 12:36:29.757250: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-05-03 12:36:29.757296: I external/local

  1/220 ━━━━━━━━━━━━━━━━━━━━ 1:51:31 31s/step - accuracy: 0.0625 - loss: 1.7301

I0000 00:00:1777811804.636502     843 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step - accuracy: 0.1881 - loss: 1.6607

2026-05-03 12:37:38.544831: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1715', 52 bytes spill stores, 52 bytes spill loads



220/220 ━━━━━━━━━━━━━━━━━━━━ 91s 275ms/step - accuracy: 0.2576 - loss: 1.5920 - val_accuracy: 0.0373 - val_loss: 1.8625
Epoch 2/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 43s 186ms/step - accuracy: 0.4054 - loss: 1.4364 - val_accuracy: 0.0419 - val_loss: 1.9950
Epoch 3/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 45s 197ms/step - accuracy: 0.4459 - loss: 1.3694 - val_accuracy: 0.5366 - val_loss: 1.4059
Epoch 4/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 44s 193ms/step - accuracy: 0.4596 - loss: 1.3335 - val_accuracy: 0.4581 - val_loss: 1.6925
Epoch 5/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 45s 196ms/step - accuracy: 0.4742 - loss: 1.2980 - val_accuracy: 0.5060 - val_loss: 1.5396
Epoch 6/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 44s 193ms/step - accuracy: 0.4803 - loss: 1.2722 - val_accuracy: 0.4667 - val_loss: 1.5432
Epoch 7/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 44s 194ms/step - accuracy: 0.4986 - loss: 1.2388 - val_accuracy: 0.2463 - val_loss: 1.9071
Epoch 8/15
220/220 ━━━━━━━━━━━━━━━━━━━━ 44s 190ms/step - accuracy: 0.5020 - loss: 1.2177 - val

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score

# Build test dataset if not already done
test_ds = tf.data.Dataset.from_tensor_slices(
    (test_df["image_path"].values, test_df["label"].values)
)

test_ds = (
    test_ds
    .map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# Predict
y_prob = model.predict(test_ds)
y_pred = np.argmax(y_prob, axis=1)
y_true = test_df["label"].values

print("Balanced accuracy:", balanced_accuracy_score(y_true, y_pred))

print(classification_report(
    y_true,
    y_pred,
    target_names=le.classes_
))

print(confusion_matrix(y_true, y_pred))

In [ ]:
import os

PROJECT_ROOT = "/home/jovyan"

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print("DATA_DIR:", DATA_DIR)
print(os.listdir(DATA_DIR))

In [ ]:
metadata_path = os.path.join(DATA_DIR, "HAM10000_metadata.csv")
metadata = pd.read_csv(metadata_path)

metadata.head()

In [ ]:
metadata["dx"].value_counts()

In [ ]:
IMG_DIR_1 = os.path.join(DATA_DIR, "HAM10000_images_part_1")
IMG_DIR_2 = os.path.join(DATA_DIR, "HAM10000_images_part_2")

print(len([f for f in os.listdir(IMG_DIR_1) if f.endswith(".jpg")]))
print(len([f for f in os.listdir(IMG_DIR_2) if f.endswith(".jpg")]))

In [ ]:
# Paths
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")

os.makedirs(MODELS_DIR, exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 7
EPOCHS = 20

In [ ]:
# Load split CSVs
train_df = pd.read_csv(os.path.join(RESULTS_DIR, "train_split.csv"))
val_df = pd.read_csv(os.path.join(RESULTS_DIR, "val_split.csv"))

In [ ]:
# Compute class weights
classes = np.sort(train_df["label"].unique())
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"]
)
class_weights = {k: min(v, 4.0) for k, v in class_weights.items()}

print("Class weights:")
for k, v in class_weights.items():
    print(f"Class {k}: {v:.4f}")


In [ ]:
# Image loader
def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = image / 255.0
    return image, label

In [ ]:
# Build datasets
train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["path"].values, train_df["label"].values)
)
val_ds = tf.data.Dataset.from_tensor_slices(
    (val_df["path"].values, val_df["label"].values)
)

train_ds = train_ds.map(load_and_preprocess_image).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(load_and_preprocess_image).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# Data augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.03),
])

In [ ]:
# Base model
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

In [ ]:
# Unfreeze top 10 layers
base_model.trainable = True
for layer in base_model.layers[:-10]:
    layer.trainable = False


In [ ]:
# Build full model
inputs = tf.keras.Input(shape=(224, 224, 3))

In [ ]:
# Apply augmentation
x = data_augmentation(inputs)

In [ ]:
# Pass through ResNet
x = base_model(x, training=False)

In [ ]:
# HEAD (your improved part)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(MODELS_DIR, "best_model.keras"),
        save_best_only=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks
)

model.save(os.path.join(MODELS_DIR, "resnet50_finetuned.keras"))
print("Fine-tuned model saved.")